# 🧪 Data check — which documents are safe to index?

Run this **before** `rag_pipeline.ipynb` whenever you add files to `data/clean/` (the cleaned PDFs made by
`clean_rag_pdfs_no_ai.ipynb`; if that folder is empty it checks `data/raw/`, like the pipeline). It reads every
document with **the same extractor the pipeline uses** and flags anything that would hurt retrieval:

| Check | Why it matters | How it's measured |
|---|---|---|
| **Exact duplicates** | the same passage is retrieved 2–3 times and fills the top-k | MD5 hash of the file |
| **Near-duplicates** | two editions of the same bulletin do the same, more subtly | share of 8-word sequences two files have in common |
| **Scanned pages** | image-only pages have no text: they are silently missing from the index | pages with < 30 characters |
| **Garbled text** | broken words embed as noise and never match a question | share of words found in a word-frequency list (`wordfreq`, Arabic + English) |
| **Internal repetition** | a file that repeats its own paragraphs crowds out other sources | share of 12-word sequences that occur more than once in the file |
| **Off-topic / policy text** | reports for governments answer farmers' questions badly but still get retrieved | density of practical farming terms vs policy terms |

Each file gets **KEEP**, **REVIEW** (read the reasons and the sample text, then decide) or **REMOVE**.
Some problems are fixed automatically by `rag_pipeline.ipynb`, so they only appear as **notes**:
exact copies are skipped, repeated passages (inside a file or shared with another file) are dropped at
chunk level, and individual unreadable pages are skipped while the rest of the file is kept.
The scores can't see everything: text from two-column pages can be interleaved (every word valid, sentences
scrambled), and exam notes or reports from other countries can be farming-related but off-target. The sample
cell in section 5 is there so you can read each flagged file before deciding.
Nothing is deleted: set `MOVE_FILES = True` in the last cell to move REMOVE files to `data/removed/`.

Needs `pip install pdfminer.six pypdf wordfreq pandas` (all in the project's `requirements.txt`).

In [1]:
import hashlib
import re
import shutil
import time
import unicodedata
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from pdfminer.high_level import extract_pages
from pdfminer.layout import LAParams, LTAnno, LTChar, LTContainer, LTTextContainer, LTTextLine
from pypdf import PdfReader
from wordfreq import zipf_frequency

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"   # cleaned PDFs (preferred, same rule as rag_pipeline.ipynb)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOCS_DIR = CLEAN_DIR if any(p.suffix.lower() in {".pdf", ".txt", ".md"} for p in CLEAN_DIR.glob("*")) else RAW_DIR
REMOVED_DIR = PROJECT_ROOT / "data" / "removed"
RESULTS_DIR = PROJECT_ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED = {".pdf", ".txt", ".md"}
EXTRACTOR = "pdfminer"   # must match rag_pipeline.ipynb
OCR_THRESHOLD = 30       # characters; below this a page counts as scanned / empty

# ---- decision thresholds (see the distribution plots-as-tables below before changing them)
GARBLED_REMOVE = 0.85        # valid-word share below this -> REMOVE
GARBLED_REVIEW = 0.92        # below this -> REVIEW
SCANNED_REMOVE = 0.60        # more than this share of pages without text -> REMOVE (needs OCR)
SCANNED_REVIEW = 0.20
NEAR_DUP = 0.80              # this share of the smaller file found in another file -> REMOVE the smaller
NEAR_DUP_REVIEW = 0.40
REPEAT_REVIEW = 0.30         # this share of a file's text repeats itself -> REVIEW
MIN_WORDS = 300              # very short files add little; flagged for REVIEW
POLICY_REVIEW = 0.30         # policy terms / (policy + practical terms) above this -> REVIEW
PRACTICAL_REVIEW = 8.0       # fewer practical farming terms per 1,000 words -> REVIEW

print("Checking:", DOCS_DIR)

Checking: D:\agri-rag-assistant\data\clean


## 1. Read every document (same extractor as the pipeline)

In [2]:
ARABIC_CHARS = "؀-ۿݐ-ݿﭐ-﷿ﹰ-﻿"   # incl. presentation forms
_ARABIC_CHAR_RE = re.compile(f"[{ARABIC_CHARS}]")
_LATIN_CHAR_RE = re.compile(r"[A-Za-z]")
_LTR_RUN = r"[A-Za-z0-9٠-٩۰-۹][A-Za-z0-9٠-٩۰-۹.,٫%/:+\-]*"
_LTR_RUNS_RE = re.compile(f"{_LTR_RUN}(?:\\s+{_LTR_RUN})*")
_SWAP_BRACKETS = str.maketrans("()[]", ")(][")


def is_rtl(text: str) -> bool:
    return len(_ARABIC_CHAR_RE.findall(text)) > len(_LATIN_CHAR_RE.findall(text))


def fragment_text(line) -> str:
    """Text of one pdfminer line fragment in reading order.

    Many Arabic PDFs draw their glyphs left to right (visual order). Such fragments are read right to
    left *glyph by glyph*: a ligature glyph that stands for several letters (لأ, تح, جم ...) keeps its
    letters in order, and runs of digits / Latin words are turned back to left-to-right.
    """
    items = [o for o in line if isinstance(o, (LTChar, LTAnno))]
    text = "".join(o.get_text() for o in items)
    if not is_rtl(text):
        return _fix_arabic_runs(items) if _ARABIC_CHAR_RE.search(text) else text
    xs = [o.x0 for o in items if isinstance(o, LTChar)]
    if sum(b > a for a, b in zip(xs, xs[1:])) >= len(xs) / 2:  # glyphs stored left to right
        text = "".join(o.get_text() for o in reversed(items))
        text = _LTR_RUNS_RE.sub(lambda m: m.group(0)[::-1], text)
    return text


def _fix_arabic_runs(items) -> str:
    """English line containing Arabic words: if the Arabic glyphs are stored left to right, reverse each
    Arabic run glyph by glyph (keeps ligature glyphs whole)."""
    def is_ar(o):
        return isinstance(o, LTChar) and bool(_ARABIC_CHAR_RE.search(o.get_text()))
    ar_x = [o.x0 for o in items if is_ar(o)]
    if len(ar_x) < 2 or sum(b > a for a, b in zip(ar_x, ar_x[1:])) < (len(ar_x) - 1) / 2:
        return "".join(o.get_text() for o in items)
    out, run = [], []
    for o in items + [None]:
        if o is not None and (is_ar(o) or (run and o.get_text() == " ")):
            run.append(o.get_text())
            continue
        trailing = []
        while run and run[-1] == " ":
            trailing.append(run.pop())
        out.extend(reversed(run))
        out.extend(trailing)
        run = []
        if o is not None:
            out.append(o.get_text())
    return "".join(out)


def _text_lines(obj):
    if isinstance(obj, LTTextLine):
        yield obj
    elif isinstance(obj, LTContainer):
        for child in obj:
            yield from _text_lines(child)


def _fix_brackets(text: str) -> str:
    """Some PDFs store mirrored brackets: on lines where ')' comes before '(', swap them."""
    def violations(t):
        depth = bad = 0
        for ch in t:
            depth += (ch == "(") - (ch == ")")
            if depth < 0:
                bad, depth = bad + 1, 0
        return bad + depth
    lines = []
    for line in text.split("\n"):  # line by line: one page can mix both conventions
        swapped = line.translate(_SWAP_BRACKETS)
        lines.append(swapped if violations(swapped) < violations(line) else line)
    return "\n".join(lines)


def _block_text(layout) -> str:
    """pdfminer's own reading: text blocks in layout order (handles columns and tables well)."""
    blocks = []
    for element in layout:
        if isinstance(element, LTTextContainer):
            lines = [re.sub(r"\s+", " ", fragment_text(l)).strip() for l in _text_lines(element)]
            lines = [l for l in lines if l]
            if lines:
                blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def _split_line_pairs(layout) -> tuple[int, int]:
    """Count touching fragments of the same printed line that pdfminer put in different blocks."""
    frags = []
    for b, element in enumerate(layout):
        if isinstance(element, LTTextContainer):
            for line in _text_lines(element):
                if line.get_text().strip():
                    frags.append((b, line))
    frags.sort(key=lambda f: (-round((f[1].y0 + f[1].y1) / 2), f[1].x0))
    pairs = 0
    rows = 1 if frags else 0
    for (b1, l1), (b2, l2) in zip(frags, frags[1:]):
        same_row = abs((l1.y0 + l1.y1) / 2 - (l2.y0 + l2.y1) / 2) <= 0.5 * min(l1.height, l2.height)
        rows += not same_row
        if same_row and b1 != b2 and l2.x0 - l1.x1 < 0.8 * max(l1.height, 1):
            pairs += 1
    return pairs, rows


def page_text_from_layout(layout) -> str:
    """Page text in reading order.

    Normally pdfminer's block reading is used. Some PDFs store one printed line as several touching
    fragments and pdfminer files them in different blocks, which scrambles sentences; on such pages
    the lines are rebuilt from fragment positions instead (right to left for Arabic), and real text
    columns, if any, are read one column at a time.
    """
    block_text = _block_text(layout)
    if not is_rtl(block_text):  # English pages: pdfminer's block reading
        return block_text
    pairs, rows = _split_line_pairs(layout)
    if pairs < max(3, 0.1 * rows):
        return _fix_brackets(block_text)
    frags = []
    for line in _text_lines(layout):
        raw = fragment_text(line).replace("\n", "")
        text = re.sub(r"\s+", " ", raw).strip()
        if text:
            sizes = [c.size for c in line if isinstance(c, LTChar)]
            frags.append({"x0": line.x0, "x1": line.x1, "y": (line.y0 + line.y1) / 2,
                          "h": max(line.height, 1.0), "size": sorted(sizes)[len(sizes) // 2] if sizes else line.height,
                          "text": text, "lead": raw[:1].isspace(), "trail": raw[-1:].isspace()})
    # sideways text (a title running down the margin) arrives as a stack of one-letter fragments: drop it
    def lone_letter(f):
        return len(f["text"]) == 1 and f["text"].isalpha()
    singles = Counter(round(f["x0"] / 3) for f in frags if lone_letter(f))
    frags = [f for f in frags if not (lone_letter(f) and singles[round(f["x0"] / 3)] >= 4)]
    if not frags:
        return ""
    rtl_page = is_rtl(" ".join(f["text"] for f in frags))

    # 1. fragments on the same baseline -> one visual line
    frags.sort(key=lambda f: -f["y"])
    rows = []
    for f in frags:
        if rows and abs(rows[-1]["y"] - f["y"]) <= 0.5 * min(rows[-1]["h"], f["h"]):
            rows[-1]["frags"].append(f)
        else:
            rows.append({"y": f["y"], "h": f["h"], "frags": [f]})

    # 2. within a line, split at wide gaps (columns / table cells) into segments
    for row in rows:
        row["frags"].sort(key=lambda f: f["x0"])
        segments, current = [], [row["frags"][0]]
        for a, b in zip(row["frags"], row["frags"][1:]):
            if b["x0"] - a["x1"] > max(0.8 * a["size"], 5):
                segments.append(current)
                current = [b]
            else:
                current.append(b)
        segments.append(current)
        row["segments"] = segments

    # 3. column gutters: an x position with wide text on both sides in many lines, and few lines crossing it
    width = max(f["x1"] for f in frags) - min(f["x0"] for f in frags)
    candidates = sorted({round((a[-1]["x1"] + b[0]["x0"]) / 2)
                         for r in rows for a, b in zip(r["segments"], r["segments"][1:])})
    gutters = []
    for x in candidates:
        crossing = sum(any(sg[0]["x0"] < x < sg[-1]["x1"] for sg in r["segments"]) for r in rows)
        column_rows = 0
        for r in rows:
            left = [sg for sg in r["segments"] if sg[-1]["x1"] <= x]
            right = [sg for sg in r["segments"] if sg[0]["x0"] >= x]
            if left and right:
                l, rr = max(left, key=lambda sg: sg[-1]["x1"]), min(right, key=lambda sg: sg[0]["x0"])
                column_rows += (l[-1]["x1"] - l[0]["x0"] > 0.2 * width) and (rr[-1]["x1"] - rr[0]["x0"] > 0.2 * width)
        if crossing <= 0.2 * len(rows) and column_rows >= max(4, 0.15 * len(rows)):
            if not gutters or x - gutters[-1] > 0.1 * width:
                gutters.append(x)

    def segment_text(seg) -> str:
        parts = sorted(seg, key=lambda f: -f["x0"]) if rtl_page else seg
        out = parts[0]["text"]
        for prev, f in zip(parts, parts[1:]):
            gap = (prev["x0"] - f["x1"]) if rtl_page else (f["x0"] - prev["x1"])
            space = prev["trail"] or f["lead"] or gap > 0.15 * f["size"]
            out += (" " if space else "") + f["text"]
        return out

    def column_of(seg) -> int:
        mid = (seg[0]["x0"] + seg[-1]["x1"]) / 2
        col = sum(mid > g for g in gutters)
        return (len(gutters) - col) if rtl_page else col

    # 4. emit in reading order; a line that crosses a gutter (title, full-width paragraph) ends a band
    out, band = [], []

    def flush():
        for col in sorted({c for c, _, _ in band}):
            out.extend(t for c, _, t in band if c == col)
            out.append("")
        band.clear()

    for row in rows:
        segs = row["segments"][::-1] if rtl_page else row["segments"]
        if gutters and not any(any(s[0]["x0"] < g < s[-1]["x1"] for g in gutters) for s in segs):
            for s in segs:
                band.append((column_of(s), row["y"], segment_text(s)))
        else:
            flush()
            out.append("   ".join(segment_text(s) for s in segs))
    flush()
    text = re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()
    return _fix_brackets(text) if rtl_page else text


def pdfminer_pages(path: Path) -> list[str]:
    return [page_text_from_layout(layout) for layout in extract_pages(str(path), laparams=LAParams())]


def pypdf_pages(path: Path) -> list[str]:
    reader = PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")  # many "protected" PDFs open with an empty password
    return [page.extract_text() or "" for page in reader.pages]


def load_document(path: Path) -> list[dict]:
    """Return [{'page': n, 'text': ...}] with 1-based page numbers."""
    if path.suffix.lower() == ".pdf":
        try:
            texts = pdfminer_pages(path) if EXTRACTOR == "pdfminer" else pypdf_pages(path)
        except Exception:  # damaged or unusual PDF: fall back to pypdf
            texts = pypdf_pages(path)
        return [{"page": i, "text": t} for i, t in enumerate(texts, 1)]
    return [{"page": 1, "text": path.read_text(encoding="utf-8", errors="ignore")}]


def file_md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

PERSIAN_TO_ARABIC = str.maketrans({"ی": "ي", "ک": "ك", "ھ": "ه", "ۀ": "ه", "ہ": "ه"})


DIACRITICS_RE = re.compile("[ؐ-ًؚ-ٰٟۖ-ۭـ]")


def normalize(text: str) -> str:
    """NFKC + no diacritics/tatweel + Persian letters -> Arabic (what the pipeline does before chunking)."""
    text = unicodedata.normalize("NFKC", text).replace("­", "")
    return DIACRITICS_RE.sub("", text).translate(PERSIAN_TO_ARABIC)


files = sorted(p for p in DOCS_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED)
other = sorted(p.name for p in DOCS_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in SUPPORTED | {""})
if other:
    print(f"Not checked (the pipeline ignores these formats): {other}")
if not files:
    raise FileNotFoundError(f"No PDF/TXT/MD files in {DOCS_DIR}")

docs, failed = {}, []
t0 = time.time()
for i, path in enumerate(files, 1):
    name = path.relative_to(DOCS_DIR).as_posix()
    try:
        pages = [normalize(p["text"]) for p in load_document(path)]
    except Exception as exc:
        failed.append({"file": name, "error": f"{type(exc).__name__}: {exc}"})
        continue
    docs[name] = {"path": path, "md5": file_md5(path), "pages": pages}
    print(f"\r{i}/{len(files)} read ({time.time() - t0:.0f}s) {name[:60]:<60}", end="")
print(f"\nRead {len(docs)} files, {sum(len(d['pages']) for d in docs.values()):,} pages "
      f"in {time.time() - t0:.0f}s. Failed: {len(failed)}")
if failed:
    display(pd.DataFrame(failed))

53/53 read (123s) نشرة-فول-الصويا-طبع_compressed.pdf                          
Read 53 files, 2,867 pages in 123s. Failed: 0


## 2. Text quality per document
A word counts as **valid** when it appears in the `wordfreq` frequency list for its language (Arabic
words of 2+ letters, English words of 3+ letters). Clean bulletins score 0.94–0.99; text with swapped
or broken letters drops quickly (`رفباير` instead of `فبراير`, `العTج` instead of `العلاج`).

In [3]:
AR_WORD_RE = re.compile("[ء-ي]{2,}")
EN_WORD_RE = re.compile("[A-Za-z]{3,}")
MIXED_RE = re.compile("[ء-ي]+[A-Za-z]+|[A-Za-z]+[ء-ي]+")   # Latin letters glued into Arabic words
ODD_RE = re.compile("[-�]")                                  # private-use / replacement chars


@lru_cache(maxsize=None)
def is_word(word: str, lang: str) -> bool:
    return zipf_frequency(word, lang) > 0


def page_stats(text: str) -> dict:
    ar = AR_WORD_RE.findall(text)
    en = [w.lower() for w in EN_WORD_RE.findall(text)]
    valid = sum(is_word(w, "ar") for w in ar) + sum(is_word(w, "en") for w in en)
    return {"words": len(ar) + len(en), "valid": valid, "ar": len(ar), "en": len(en),
            "mixed": len(MIXED_RE.findall(text)), "odd": len(ODD_RE.findall(text)), "chars": len(text.strip())}


rows = []
for name, d in docs.items():
    stats = [page_stats(p) for p in d["pages"]]
    d["page_stats"] = stats
    words = sum(s["words"] for s in stats)
    text_pages = [s for s in stats if s["chars"] >= OCR_THRESHOLD]
    garbled_pages = [s for s in stats if s["words"] >= 40 and s["valid"] / s["words"] < 0.80]
    ar, en = sum(s["ar"] for s in stats), sum(s["en"] for s in stats)
    rows.append({
        "file": name,
        "pages": len(stats),
        "scanned_share": round(1 - len(text_pages) / max(len(stats), 1), 2),
        "language": "ar" if ar > 2 * en else "en" if en > 2 * ar else "mixed",
        "words": words,
        "valid_words": round(sum(s["valid"] for s in stats) / max(words, 1), 3),
        "garbled_pages": len(garbled_pages),
        "odd_per_1k": round(1000 * sum(s["odd"] + s["mixed"] for s in stats) / max(words, 1), 1),
    })
quality = pd.DataFrame(rows).set_index("file")
quality.sort_values("valid_words")

,pages,scanned_share,language,words,valid_words,garbled_pages,odd_per_1k
file,,,,,,,
نشرة-الكوسة.pdf,24,0.0,ar,5011,0.859,0,0.2
AGRI_021.pdf,133,0.0,ar,41644,0.879,41,0.1
NEW_rice_diseases.pdf,2,0.0,ar,156,0.891,0,0.0
res4_-mohamed-abdel-hady-sayed-abdel-wahab.pdf,37,0.0,en,3495,0.910,3,0.0
917873240-Key-Fact.pdf,286,0.0,en,40043,0.937,18,0.0
NEW_onion.pdf,22,0.0,ar,4390,0.943,0,0.0
التداول-الأمثل-لمحصول-البصل.pdf,3,0.0,ar,531,0.944,0,0.0
الكانتلوب.pdf,27,0.0,ar,5486,0.946,1,0.5
زراعة-وانتاج-الخيار.pdf,27,0.0,ar,6141,0.956,0,0.0


## 3. Duplicates and near-duplicates

In [4]:
# Exact copies: same bytes. Keep the copy with the most descriptive file name.
GENERIC = {"new", "copy", "compressed", "final", "pdf", "agri", "file", "ticket"}


def name_score(name: str) -> int:
    words = re.findall(r"[^\W\d_]+", Path(name).stem.lower())
    return sum(len(w) for w in words if w not in GENERIC)


exact_drop = {}
by_hash = defaultdict(list)
for name, d in docs.items():
    by_hash[d["md5"]].append(name)
for names in by_hash.values():
    if len(names) > 1:
        keep = max(names, key=name_score)
        for n in names:
            if n != keep:
                exact_drop[n] = keep
print(f"{len(exact_drop)} exact duplicate file(s):")
for n, keep in exact_drop.items():
    print(f"  {n}  ==  {keep}  (kept)")

1 exact duplicate file(s):
  NEW_sugar_beet.pdf  ==  1-نشرة-بنجر-السكر_compressed.pdf  (kept)


In [5]:
# Near-duplicates: share of 8-word sequences of the smaller file that also appear in the other file.
def match_form(text: str) -> str:
    text = re.sub("[آأإٱ]", "ا", text.lower())
    return text.replace("ى", "ي").replace("ة", "ه")


def shingles(text: str, n: int = 8) -> set[int]:
    words = re.findall(r"\w+", match_form(text))
    return {hash(" ".join(words[i:i + n])) for i in range(max(len(words) - n + 1, 0))}


sh = {name: shingles("\n".join(d["pages"])) for name, d in docs.items() if name not in exact_drop}
pairs = []
names = [n for n in sh if len(sh[n]) >= 50]
for i, a in enumerate(names):
    for b in names[i + 1:]:
        small, big = (a, b) if len(sh[a]) <= len(sh[b]) else (b, a)
        shared = len(sh[small] & sh[big]) / len(sh[small])
        if shared >= NEAR_DUP_REVIEW:
            pairs.append({"smaller_file": small, "larger_file": big, "shared": round(shared, 2)})
near = pd.DataFrame(pairs, columns=["smaller_file", "larger_file", "shared"]).sort_values("shared", ascending=False)
near

,smaller_file,larger_file,shared
0,نشرة-الذرة-نهائى-copy_compressed.pdf,NEW_maize.pdf,0.68
2,القمح-فى-الأراضى-الجديدة_compressed.pdf,NEW_wheat_new.pdf,0.59
4,القمح-قديم-2022_compressed.pdf,القمح-فى-الأراضى-الجديدة_compressed.pdf,0.59
1,نشرة-االقصب-2023-copy_compressed.pdf,NEW_sugarcane_seedlings.pdf,0.52
3,القمح-قديم-2022_compressed.pdf,NEW_wheat_new.pdf,0.46


## 4. Internal repetition and topic

In [6]:
def repeated_share(pages: list[str]) -> float:
    """Share of 12-word sequences that occur more than once in the same file (headers/footers removed)."""
    line_pages = Counter()
    for p in pages:
        line_pages.update({l.strip() for l in p.splitlines() if l.strip()})
    header_lines = {l for l, c in line_pages.items() if len(pages) >= 4 and c / len(pages) >= 0.3}
    body = " ".join(l for p in pages for l in p.splitlines() if l.strip() and l.strip() not in header_lines)
    words = re.findall(r"\w+", match_form(body))
    grams = [" ".join(words[i:i + 12]) for i in range(max(len(words) - 11, 0))]
    if not grams:
        return 0.0
    counts = Counter(grams)
    return sum(c for c in counts.values() if c > 1) / len(grams)


PRACTICAL = [  # practical farming vocabulary (matched as substrings of normalised text)
    "فدان", "كجم", "تسميد", "سماد", "اسمده", "الري", "ريه", "تقاوي", "تقاوى", "حصاد", "الزراعه", "شتلات", "مبيد",
    "افات", "الصنف", "اصناف", "التربه", "الرش", "المحصول", "حشائش", "الجوره", "الخطوط", "الاشجار", "الثمار",
    "fertiliz", "irrigat", "sowing", "seed", "harvest", "pest", "yield", "soil", "crop", "planting", "spray",
    "weed", "feddan", "hectare", "manure", "variety", "nursery", "pruning", "insect", "disease",
]
POLICY = [
    "السياسات", "الحكومات", "الفقر", "التنميه المستدامه", "الامن الغذائي", "استراتيجي", "المؤسسات", "اصحاب المصلحه",
    "التغذيه", "الاطار", "المادة", "policy", "policies", "government", "poverty", "stakeholder", "framework",
    "governance", "institution", "article", "investment", "countries", "national",
]


def term_count(text: str, terms: list[str]) -> int:
    return sum(text.count(t) for t in terms)


topic_rows = []
for name, d in docs.items():
    body = match_form(" ".join(d["pages"]))
    practical, policy = term_count(body, PRACTICAL), term_count(body, POLICY)
    words = max(quality.loc[name, "words"], 1)
    topic_rows.append({
        "file": name,
        "repeated": round(repeated_share(d["pages"]), 2),
        "practical_per_1k": round(1000 * practical / words, 1),
        "policy_share": round(policy / max(practical + policy, 1), 2),
    })
topic = pd.DataFrame(topic_rows).set_index("file")
report = quality.join(topic)
report[["words", "repeated", "practical_per_1k", "policy_share"]].sort_values("practical_per_1k")

,words,repeated,practical_per_1k,policy_share
file,,,,
135905761430.pdf,16678,0.12,2.5,0.93
Unit 1.pdf,12303,0.01,10.6,0.40
2-البحث-الثاني.pdf,3052,0.03,11.5,0.10
AGRI_021.pdf,41644,0.01,13.6,0.52
cd7067en.pdf,53458,0.06,17.7,0.49
التداول-الأمثل-لمحصول-البصل.pdf,531,0.00,30.1,0.00
917873240-Key-Fact.pdf,40043,0.00,36.9,0.12
ticket_file6214bd8f64479-Pomme-de-terre-Ardna (1).pdf,1633,0.00,38.6,0.02
AGRI_011_PDF.pdf,8544,0.02,42.1,0.35


## 5. Verdict

In [7]:
near_drop = {}
for _, r in near.iterrows():
    if r.shared >= NEAR_DUP and r.smaller_file not in near_drop and r.larger_file not in near_drop:
        near_drop[r.smaller_file] = (r.larger_file, r.shared)


def verdict(name: str, r: pd.Series) -> tuple[str, list[str], list[str]]:
    remove, review, notes = [], [], []
    if name in exact_drop:
        remove.append(f"exact copy of {exact_drop[name]} (the pipeline skips it anyway)")
    if name in near_drop:
        notes.append(f"{near_drop[name][1]:.0%} of its text is also in {near_drop[name][0]}: those passages are dropped when indexing")
    if r.scanned_share > SCANNED_REMOVE:
        remove.append(f"{r.scanned_share:.0%} of pages have no text (scanned: needs OCR)")
    elif r.scanned_share > SCANNED_REVIEW:
        review.append(f"{r.scanned_share:.0%} of pages have no text")
    if r.words and r.valid_words < GARBLED_REMOVE:
        remove.append(f"garbled text: only {r.valid_words:.0%} valid words")
    elif r.words and r.valid_words < GARBLED_REVIEW:
        review.append(f"partly garbled: {r.valid_words:.0%} valid words")
    if r.garbled_pages and not remove:
        notes.append(f"{r.garbled_pages} unreadable page(s) will be skipped, the rest is indexed")
    for _, p in near[(near.smaller_file == name) & (near.shared < NEAR_DUP)].iterrows():
        notes.append(f"{p.shared:.0%} overlap with {p.larger_file}: shared passages are dropped when indexing")
    if r.repeated > REPEAT_REVIEW:
        notes.append(f"{r.repeated:.0%} of its text repeats itself: repeats are dropped when indexing")
    if r.words < MIN_WORDS and not remove:
        review.append(f"very little text ({r.words} words)")
    if r.policy_share > POLICY_REVIEW:
        review.append(f"policy/report language ({r.policy_share:.0%} policy terms)")
    elif r.practical_per_1k < PRACTICAL_REVIEW:
        review.append(f"few practical farming terms ({r.practical_per_1k}/1k words)")
    if remove:
        return "REMOVE", remove + review, notes
    return ("REVIEW" if review else "KEEP"), review, notes


report[["verdict", "reasons", "notes"]] = [(v, "; ".join(rs), "; ".join(ns))
                                           for v, rs, ns in (verdict(n, r) for n, r in report.iterrows())]
order = {"REMOVE": 0, "REVIEW": 1, "KEEP": 2}
report = report.sort_values(by="verdict", key=lambda s: s.map(order))
report.to_csv(RESULTS_DIR / "data_check.csv", encoding="utf-8-sig")
(RESULTS_DIR / "files_to_remove.txt").write_text(
    "\n".join(report.index[report.verdict == "REMOVE"]) + "\n", encoding="utf-8")

print(report.verdict.value_counts().to_string())
report[["verdict", "reasons", "notes", "pages", "language", "valid_words", "scanned_share", "repeated", "practical_per_1k", "policy_share"]]

verdict
KEEP      44
REVIEW     8
REMOVE     1


,verdict,reasons,notes,pages,language,valid_words,scanned_share,repeated,practical_per_1k,policy_share
file,,,,,,,,,,
NEW_sugar_beet.pdf,REMOVE,exact copy of 1-نشرة-بنجر-السكر_compressed.pdf (the pipeline skips it anyway),,26,ar,0.973,0.0,0.00,57.1,0.00
135905761430.pdf,REVIEW,policy/report language (93% policy terms),,96,en,0.991,0.0,0.12,2.5,0.93
AGRI_021.pdf,REVIEW,partly garbled: 88% valid words; policy/report language (52% policy terms),"41 unreadable page(s) will be skipped, the rest is indexed",133,ar,0.879,0.0,0.01,13.6,0.52
cd7067en.pdf,REVIEW,policy/report language (49% policy terms),"3 unreadable page(s) will be skipped, the rest is indexed",297,en,0.981,0.0,0.06,17.7,0.49
res4_-mohamed-abdel-hady-sayed-abdel-wahab.pdf,REVIEW,partly garbled: 91% valid words,"3 unreadable page(s) will be skipped, the rest is indexed",37,en,0.910,0.0,0.02,51.8,0.02
Unit 1.pdf,REVIEW,policy/report language (40% policy terms),,63,en,0.998,0.0,0.01,10.6,0.40
NEW_rice_diseases.pdf,REVIEW,partly garbled: 89% valid words; very little text (156 words),,2,ar,0.891,0.0,0.00,128.2,0.00
AGRI_011_PDF.pdf,REVIEW,policy/report language (35% policy terms),,44,en,0.992,0.0,0.02,42.1,0.35
نشرة-الكوسة.pdf,REVIEW,partly garbled: 86% valid words,,24,ar,0.859,0.0,0.01,55.7,0.00


In [8]:
# Read before deciding: a sample from the middle of every REVIEW / REMOVE file
for name, r in report[report.verdict != "KEEP"].iterrows():
    pages = [p for p in docs[name]["pages"] if len(p.strip()) >= 200]
    sample = pages[len(pages) // 2][:500].replace("\n", " ") if pages else "(no text)"
    display(Markdown(f"**{r.verdict} · {name}** — {r.reasons}\n\n> {sample}"))

**REMOVE · NEW_sugar_beet.pdf** — exact copy of 1-نشرة-بنجر-السكر_compressed.pdf (the pipeline skips it anyway)

> .25 أعراض نقص المنجنيز  تظهر أعراض نقص المنجنيز بوجود بقع صفراء شاحبة ا لون علي بعض أجزاء من الأوراق بين العروق  (الأوعية) وتبقى العروق فقط بلون أخضر شاحب.  فى حالة النقص الشديد يتحول لون البقع الصفراء إلى الأصفر البرتقالى ، وتنمو الأوراق قائمة لأعلى  وتلتف حوافها للداخل.  تظهر أعراض النقص من البادرة وحتى الحصاد.  تحتوى معظم الأراضى على كميات كافية من المنجنيز الميسر الذى يحتاجه بنجر السكر بكميات قليلة ،  وبالتالى لا تظهر أعراض نقصه غالبا .  يقل تيسر المنجنيز للنباتات بزيادة درجة الحموضة (خاصة ف

**REVIEW · 135905761430.pdf** — policy/report language (93% policy terms)

> small farmers.  A more complex environment  The environment in which agricultural research and extension must operate today, is  increasingly more complex than those of the Green Revolution.  Decreasing public funding for agricultural research and extension  During the 1960’s and 70’s, investment in research grew at a rapid pace, estimated at  a  global average of over 6% per year (Alston et al., 1998; Pardey and Alston, 1995).  During  the 1980’s, investment stagnated or declined, in particular

**REVIEW · AGRI_021.pdf** — partly garbled: 88% valid words; policy/report language (52% policy terms)

> الت كيف الذاتي في مقابل المخطط: ستستجيب المجتمعات وصرغار منتجري الأغذيرة بشركل ذاتري  لتغيرر المنراخ الحالي والمتوقع. وتتيح بنوك البذور والحبروب المحليرة آليرات لحفرظ ونقرل المعرارف  ال تجريبيرة ذات القيمرة للتكيف الذاتي. وبدلا من ذلك، يمكن أن تضرع الحكومرة أو وكرالات أخررى  اسرتراتيجيات للتكيرف في إطرار برنامج تخطيطي. والمرجح أن تكون النهج التي تجمع الإستراتيجيتين  أكفأ نهج.  إقر ار قيمة المعارف المحلية القائمة، وتنفيذ أفضل الممارسات، وتوليد معارف جديرد ة: يمكرن برذل  الكرثير مرن أجل التكيف مع 

**REVIEW · cd7067en.pdf** — policy/report language (49% policy terms)

> NOTES: “Production” refers to the contribution of farms to domestic agricultural  production (including exports and excluding imports). “Consumption” refers to the  source and farm patterns of agricultural output destined for regional consumption  (domestic and non-domestic).  SOURCE (text and figures): Adapted from Taherzadeh, O., Cai, H. & Mogollón, J.  (forthcoming). The hidden role of small-scale farmers in global food  then they merit supportive policies to increase food production; such su

**REVIEW · res4_-mohamed-abdel-hady-sayed-abdel-wahab.pdf** — partly garbled: 91% valid words

> kg N fed-l for anhydrous ammonia as a source of NH4* led to the increase of grain  yield, 1000 grains and  the other nitrogen sources.  tt:  Abdel lVahab , M. A.S. et aL  grains increased but it was noticeable in the case of the combined featments. The  highest values were obtained  with the combined treatments of ( 12.5 kg NOr-N / 87'5  kg NHa-N ) and gave39.87, 12.V2, 42.21 and229.2Kg  fed-r for N,P ,K and protein yield, respectively as it  Metwally (2009) found that a positive effect of N-  f

**REVIEW · Unit 1.pdf** — policy/report language (40% policy terms)

> Summary of social change  1. Social change involves the transformation of cultural norms and values,  behavior, social institutions, and social structure. As societies become more  modern, they become larger, more heterogeneous, and more impersonal, and  their sense of community declines. Traditions decline as well, while individual  freedom of thought and behavior increases. Some sociologists view modernization  positively, while others view it negatively.  2. A functionalist understanding of s

**REVIEW · NEW_rice_diseases.pdf** — partly garbled: 89% valid words; very little text (156 words)

> العلاج: نقع التقاوي في محلول ملحي لمدة 48 ساعة، ثم غسيل التقاوي بالماء العذب.  .4 البكانا  .5 التبقع (الصورة الثانية في الملصق)  العلاج: نستخدم الكوبرال %50 بمعدل 1 كجم للفدان.  .6 ملخصات مرجعية سريعة  التبقع: الري والصرف والتسميد المتوازن + رش كبريتات الزنك 2 كجم/فدان  اللفحة: فوجي ون 400 سم/200 لتر ماء (3 رشات) أو البيم 100 جم/200 لتر ماء (رشتان)  أطراف الأوراق النيماتودي: الفيوردان أو الموكاب 6 كجم/فدان  البكانا: نقع التقاوي في محلول ملحي 48 ساعة ثم غسلها بالماء العذب  التبقع (الحالة الثانية)

**REVIEW · AGRI_011_PDF.pdf** — policy/report language (35% policy terms)

> exposure to pesticides; 5.2.4.4using returnable and refilable containers where effective  container collection systems are in place; 5.2.4.5using containers that are not  attractive for subsequent reuse and promoting programmes to discour- age their  reuse, where effective container collection systems are not in place; ment from the  possible adverse effects of pesticides.  5.4 Entities addressed by the Code should consider all available facts and should  promote responsible information dissemin

**REVIEW · نشرة-الكوسة.pdf** — partly garbled: 86% valid words

> جمع الأوراق المسابة و اإعدامها ، عدم الإسراف في التسميد الزوتي والهتمام بالتسميد الفوسفاتي و  المكافحة الزراعية والميكانيكية  البوتاسي .  المكافحة الحيوية  المبيدات الحيوية  روماكتين %EC 1.8 بمعدل 40 سم100 / 3 لتر ماء فيرتيميك %EC 1.8 بمعدل 40 سم100 / 3 لتر ماء  سوبركين %EC 1.8 بمعدل 40 سم100 / 3 لتر ماء  اكتارا %WG 25 بمعدل 20 جم / 100 لتر ماء موسبيلان %SP 20 بمعدل 25 جم / 100 لتر ماء كوماندو  %SC 35 بمعدل 75 سم100 / 3 لتر ماء كونفيديت %SC 35 بمعدل 75 سم100 / 3 لتر ماء اكتيلك EC  %50 بمعل 375 س

## 6. Move the files you decided to drop
Edit `DROP` if you disagree with a verdict (add REVIEW files you want gone, remove ones you want to keep),
then set `MOVE_FILES = True`. Files are **moved** to `data/removed/`, not deleted, so you can undo it.
After moving, re-run `rag_pipeline.ipynb` (**Restart & Run All**) to rebuild the vector store.

In [ ]:
DROP = list(report.index[report.verdict == "REMOVE"])
MOVE_FILES = False

print("Will move:" if MOVE_FILES else "Would move (set MOVE_FILES = True to do it):")
for name in DROP:
    print("  ", name)
if MOVE_FILES:
    for name in DROP:
        target = REMOVED_DIR / name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(DOCS_DIR / name), str(target))
    print(f"Moved {len(DROP)} file(s) to {REMOVED_DIR}")